# 56. PDF Files — `reportlab · pypdf`
**Python Libraries Lab · Phase 3 · Files & Automation**

_Generate invoices as PDFs, then merge, split, stamp and read them._

✅ **Runs in Colab** as-is.

PDF is the format for documents that must look the same everywhere, like invoices, tickets and reports. ReportLab creates PDFs from Python, with text, tables, images and charts. pypdf works with existing PDFs. It reads text out of them, merges and splits files, rotates pages, adds watermarks and adds passwords.

**In this notebook:** You will generate real ShopEasy invoices with a header, customer details, an items table and totals, create one per order, merge them into a single file, stamp a PAID watermark, extract text back out, split pages, and protect a file with a password.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q reportlab pypdf
import importlib.metadata as md
for pkg in ['reportlab', 'pypdf']:
    print(pkg, md.version(pkg))

## Step 1 · An invoice with Platypus
Paragraphs and a table flow onto the page. `TableStyle` sets borders and colours.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

styles = getSampleStyleSheet()

def make_invoice(path, order_id, customer, city, items):
    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm, topMargin=18 * mm)
    rows = [["Item", "Qty", "Price", "Amount"]] + [[n, q, f"Rs. {p:,}", f"Rs. {p * q:,}"] for n, q, p in items]
    total = sum(p * q for _, q, p in items)
    rows += [["", "", "GST 18%", f"Rs. {total * 0.18:,.2f}"], ["", "", "Total", f"Rs. {total * 1.18:,.2f}"]]
    table = Table(rows, colWidths=[80 * mm, 20 * mm, 35 * mm, 35 * mm])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2C5A9A")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("ALIGN", (1, 0), (-1, -1), "RIGHT"), ("GRID", (0, 0), (-1, -3), 0.4, colors.grey),
        ("FONTNAME", (0, -1), (-1, -1), "Helvetica-Bold"), ("LINEABOVE", (2, -1), (-1, -1), 1, colors.black),
    ]))
    doc.build([Paragraph("ShopEasy", styles["Title"]), Paragraph(f"Tax invoice <b>INV-{order_id}</b>", styles["Heading2"]),
               Paragraph(f"Billed to: {customer}, {city}", styles["Normal"]), Spacer(1, 8 * mm), table,
               Spacer(1, 10 * mm), Paragraph("Thank you for shopping with us. Returns are free within 30 days.", styles["Italic"])])
    return total * 1.18

amount = make_invoice("invoice_1001.pdf", 1001, "Asha K.", "Pune", [("Steel Bottle", 2, 499), ("Notebook", 3, 149)])
print(f"Created invoice_1001.pdf for Rs. {amount:,.2f}")

## Step 2 · Many invoices, then merge
Generate one PDF per order, then combine them with `PdfWriter`.

In [ ]:
from pypdf import PdfReader, PdfWriter
orders = [(1002, "Ravi S.", "Delhi", [("Headphones", 1, 2499)]),
          (1003, "Meera P.", "Pune", [("Yoga Mat", 1, 1299), ("Water Jug", 2, 349)])]
for o in orders:
    make_invoice(f"invoice_{o[0]}.pdf", *o)

merged = PdfWriter()
for oid in [1001, 1002, 1003]:
    merged.append(f"invoice_{oid}.pdf")
merged.write("all_invoices.pdf")
print("all_invoices.pdf has", len(PdfReader("all_invoices.pdf").pages), "pages")

## Step 3 · Stamp a watermark
Draw 'PAID' on a canvas, then lay it over every page.

In [ ]:
from reportlab.pdfgen import canvas
c = canvas.Canvas("paid_stamp.pdf", pagesize=A4)
c.setFont("Helvetica-Bold", 90); c.setFillColor(colors.Color(0.1, 0.6, 0.3, alpha=0.25))
c.translate(300, 420); c.rotate(30); c.drawCentredString(0, 0, "PAID")
c.save()

stamp = PdfReader("paid_stamp.pdf").pages[0]
out = PdfWriter()
for page in PdfReader("all_invoices.pdf").pages:
    page.merge_page(stamp)
    out.add_page(page)
out.write("all_invoices_paid.pdf")
print("Watermarked", len(out.pages), "pages")

## Step 4 · Read text, split and rotate
Extract text to check totals; save page 2 alone, rotated.

In [ ]:
reader = PdfReader("all_invoices_paid.pdf")
text = reader.pages[1].extract_text()
print(text[:220])
import re
print("Totals found:", re.findall(r"Total Rs\. ([\d,\.]+)", " ".join(p.extract_text() for p in reader.pages)))

single = PdfWriter(); single.add_page(reader.pages[1]); single.pages[0].rotate(90)
single.write("invoice_1002_rotated.pdf")

## Step 5 · Password-protect a PDF
Encrypt so only people with the password can open it.

In [ ]:
secure = PdfWriter(clone_from="invoice_1001.pdf")
secure.encrypt(user_password="asha1001", algorithm="AES-256")
secure.write("invoice_1001_locked.pdf")
locked = PdfReader("invoice_1001_locked.pdf")
print("Encrypted:", locked.is_encrypted, "| unlock:", locked.decrypt("asha1001").name)
print(locked.pages[0].extract_text()[:40])

## ✍️ Your turn
Open `all_invoices.pdf` with `PdfReader` and store the **number of pages** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 3, f"Expected 3, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = len(PdfReader("all_invoices.pdf").pages)

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Canvas**
- **Platypus (flowables)**
- **Table**
- **PdfReader / PdfWriter**
- **Merge / split**
- **Encrypt**

**Next:** 57. Word & PowerPoint (`python-docx · python-pptx`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/57_python_docx_pptx.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install reportlab pypdf
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.